# 01 · Entrada de dados, ETL e engenharia de atributos

**Etapas do fluxo:** 2 (entrada e versionamento de dados novos) e 3 (ETL, validação e engenharia de atributos).
Inclui também o **monitoramento de drift** (critério 10).

Problema: prever se o cliente contratará o depósito a prazo (`y`) oferecido por campanhas de telemarketing de um banco
português (UCI *Bank Marketing*, `bank-additional-full.csv`, licença CC BY 4.0). Unidade de observação: um contato da campanha.

Cada ciclo:
1. recebe um `cycle_id` e incorpora o próximo lote da fila `data/raw/lotes/` (simulação da chegada de dados novos);
2. valida o lote (contrato de dados) e **interrompe o ciclo** se algo estiver fora do esperado;
3. limpa, cria atributos e divide o pool em treino (80%) e validação (20%) por **hash do `registro_id`** - cada
   registro fica sempre do mesmo lado, então o modelo vigente nunca treinou com a validação de ciclos futuros;
4. mede o drift do lote novo em relação aos dados já incorporados.

In [1]:
import sys
import warnings
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import json
import matplotlib.pyplot as plt
import pandas as pd

from src.config import caminho, carregar_config

pd.set_option("display.max_columns", 50)
cfg = carregar_config(RAIZ)

In [2]:
from src import ingestao
from src.ciclo import atualizar_contexto, diretorio_ciclo, obter_cycle_id, salvar_json

cycle_id = obter_cycle_id(cfg, criar=True)
pasta_ciclo = diretorio_ciclo(cfg, cycle_id)

estado = ingestao.carregar_estado(cfg)
lotes_novos = ingestao.selecionar_lotes_novos(cfg, estado)
lotes_anteriores = list(estado["lotes_incorporados"])
lotes_pool = lotes_anteriores + lotes_novos
data_version = ingestao.calcular_data_version(cfg, lotes_pool)

print("cycle_id         :", cycle_id)
print("lotes já no pool :", lotes_anteriores or "nenhum (ciclo inicial)")
print("lotes novos      :", lotes_novos or "nenhum - o ciclo reprocessa os dados atuais")
print("data_version     :", data_version)

cycle_id         : c20261123T121302Z-gh45172042466
lotes já no pool : nenhum (ciclo inicial)
lotes novos      : ['lote_001.csv']
data_version     : v001-a59ff626


## Validação dos dados (contrato)
Colunas esperadas, alvo binário, ids únicos, faixas plausíveis de idade e de número de contatos e limite de ausentes.
Qualquer reprovação lança `ErroQualidadeDados`: o notebook falha, o workflow para e **nenhuma promoção acontece**.

In [3]:
pool_bruto, ids_repetidos = ingestao.montar_pool(cfg, lotes_pool)
checagens = ingestao.validar_qualidade(pool_bruto, cfg, "pool")
for lote in lotes_novos:
    checagens += ingestao.validar_qualidade(ingestao.ler_lote(cfg, lote), cfg, lote)
teste_bruto = ingestao.ler_teste(cfg)
checagens += ingestao.validar_qualidade(teste_bruto, cfg, "teste")

tabela_checagens = pd.DataFrame(checagens)
salvar_json(pasta_ciclo / "qualidade_dados.json", checagens)
display(tabela_checagens)
ingestao.garantir_qualidade(checagens)
print(f"Validação aprovada. Registros repetidos entre lotes descartados: {ids_repetidos}")

,conjunto,regra,ok,detalhe
0,pool,colunas_esperadas,True,faltantes=[]
1,pool,nao_vazio,True,linhas=25948
2,pool,alvo_binario,True,"valores=['no', 'yes']"
3,pool,id_unico,True,duplicados=0
4,pool,idade_no_intervalo,True,min=18 max=88
5,pool,campaign_no_intervalo,True,min=1 max=56
6,pool,ausentes_toleraveis,True,maior fração=0.000 (age)
7,lote_001.csv,colunas_esperadas,True,faltantes=[]
8,lote_001.csv,nao_vazio,True,linhas=25948
9,lote_001.csv,alvo_binario,True,"valores=['no', 'yes']"


Validação aprovada. Registros repetidos entre lotes descartados: 0


## Diagnóstico antes do tratamento
O dataset não tem `NaN`, mas usa o valor `unknown` em atributos categóricos e o código `pdays = 999` para "nunca contatado".

In [4]:
from src.preprocessamento import diagnosticar, dividir_por_hash, engenharia_atributos, limpar

diagnostico_antes = diagnosticar(pool_bruto)
print(json.dumps(diagnostico_antes, indent=2, ensure_ascii=False))

fig, eixos = plt.subplots(1, 2, figsize=(12, 3.5))
pd.Series(diagnostico_antes["unknown"]).sort_values().plot.barh(ax=eixos[0], title="Valores 'unknown' por atributo")
(pool_bruto.groupby("lote")["y"].apply(lambda s: (s == "yes").mean())).plot.bar(ax=eixos[1], title="Taxa de 'yes' por lote")
plt.tight_layout()
plt.show()

{
  "linhas": 25948,
  "colunas": 23,
  "duplicatas_conteudo": 6,
  "ausentes": {},
  "unknown": {
    "job": 222,
    "marital": 41,
    "education": 1074,
    "default": 6612,
    "housing": 625,
    "loan": 625
  },
  "outliers_iqr": {
    "age": 35,
    "campaign": 1859,
    "pdays": 49,
    "previous": 897,
    "emp.var.rate": 4339,
    "cons.price.idx": 0,
    "cons.conf.idx": 0,
    "euribor3m": 4270,
    "nr.employed": 1038,
    "duration": 1996
  },
  "pdays_sem_contato": 25899,
  "taxa_positiva": 0.0563
}


## Limpeza e engenharia de atributos
Tratamentos **sem estado** (não aprendem parâmetros dos dados, então não vazam informação):

| Tratamento | Justificativa |
|---|---|
| remover `duration` | só é conhecida após a ligação terminar - usar seria vazamento do alvo |
| remover duplicatas de conteúdo | registros idênticos inflariam o peso de alguns contatos |
| manter `unknown` como categoria | a falta de informação é informativa (ex.: `default`) |
| `pdays=999` → `contatado_antes` + `pdays_dias` (ausente) | 999 é um código, não uma quantidade de dias |
| `campaign_log = log1p(campaign)` | cauda longa (outliers de muitas ligações) sem descartar registros |
| `qtd_desconhecidos`, `faixa_etaria` | resumem incompletude do cadastro e perfil etário |

Duplicatas são procuradas em **todas** as colunas originais (antes de remover `duration`): sem ela, clientes com o
mesmo perfil contatados no mesmo dia pareceriam repetidos e seriam descartados indevidamente.

Imputação, padronização e one-hot são **ajustados só no treino**, dentro do `Pipeline`/`ColumnTransformer` (notebooks 02 e 03).

In [5]:
def preparar(df):
    limpo, relatorio = limpar(df, cfg)
    return engenharia_atributos(limpo), relatorio

pool, relatorio_limpeza = preparar(pool_bruto)
teste, relatorio_teste = preparar(teste_bruto)
treino, validacao = dividir_por_hash(pool, cfg["particionamento"]["fracao_validacao"], cfg["dados"]["id"])

diagnostico_depois = {
    "limpeza_pool": relatorio_limpeza,
    "limpeza_teste": relatorio_teste,
    "linhas_pool": len(pool),
    "treino": {"linhas": len(treino), "ids": [int(treino.registro_id.min()), int(treino.registro_id.max())], "taxa_positiva": round(treino.y.mean(), 4)},
    "validacao": {"linhas": len(validacao), "ids": [int(validacao.registro_id.min()), int(validacao.registro_id.max())], "taxa_positiva": round(validacao.y.mean(), 4)},
    "teste": {"linhas": len(teste), "taxa_positiva": round(teste.y.mean(), 4)},
}
print(json.dumps(diagnostico_depois, indent=2, ensure_ascii=False))
pool[cfg["features"]["numericas"]].describe().T.round(2)

{
  "limpeza_pool": {
    "colunas_removidas": [
      "duration"
    ],
    "duplicatas_removidas": 6
  },
  "limpeza_teste": {
    "colunas_removidas": [
      "duration"
    ],
    "duplicatas_removidas": 0
  },
  "linhas_pool": 25942,
  "treino": {
    "linhas": 20715,
    "ids": [
      1,
      28865
    ],
    "taxa_positiva": 0.0566
  },
  "validacao": {
    "linhas": 5227,
    "ids": [
      11,
      28859
    ],
    "taxa_positiva": 0.0551
  },
  "teste": {
    "linhas": 4119,
    "taxa_positiva": 0.1126
  }
}


,count,mean,std,min,25%,50%,75%,max
age,25942.0,40.19,9.34,18.00,33.00,39.00,47.00,88.00
campaign_log,25942.0,1.15,0.52,0.69,0.69,1.10,1.39,4.04
pdays_dias,49.0,5.02,2.67,0.00,3.00,5.00,6.00,11.00
previous,25942.0,0.04,0.19,0.00,0.00,0.00,0.00,3.00
emp.var.rate,25942.0,1.00,0.75,-1.80,1.10,1.40,1.40,1.40
cons.price.idx,25942.0,93.81,0.42,92.76,93.44,93.92,93.99,94.46
cons.conf.idx,25942.0,-39.80,3.37,-50.00,-42.70,-41.80,-36.40,-36.10
euribor3m,25942.0,4.68,0.71,1.41,4.86,4.96,4.96,5.04
nr.employed,25942.0,5208.83,28.26,5099.10,5191.00,5228.10,5228.10,5228.10
qtd_desconhecidos,25942.0,0.35,0.60,0.00,0.00,0.00,1.00,5.00


## Monitoramento de drift (critério 10)
**Problema:** com dados chegando em lotes, a distribuição dos clientes e do cenário econômico muda; um modelo pode
degradar sem ninguém perceber. **Solução:** a cada ciclo, o PSI compara o lote novo com os dados já incorporados.
Se `min_features_alerta` ou mais atributos passarem do limiar, o ciclo emite **alerta** (registrado no MLflow, no
resumo do workflow e em uma *issue* aberta automaticamente no GitHub).

In [6]:
from src.drift import markdown_drift, relatorio_drift

drift_resumo = {"avaliado": False, "alerta": False, "motivo": "ciclo inicial ou sem lote novo"}
if lotes_anteriores and lotes_novos:
    referencia = pool[pool["lote"].isin([l.removesuffix(".csv") for l in lotes_anteriores])]
    novo = pool[pool["lote"].isin([l.removesuffix(".csv") for l in lotes_novos])]
    tabela_drift, drift_resumo = relatorio_drift(
        referencia, novo, cfg["features"]["numericas"], cfg["features"]["categoricas"], cfg["drift"]
    )
    drift_resumo.update({"avaliado": True, "taxa_positiva_referencia": round(referencia.y.mean(), 4), "taxa_positiva_novo": round(novo.y.mean(), 4)})
    tabela_drift.to_csv(pasta_ciclo / "drift.csv", index=False)
    (pasta_ciclo / "drift.md").write_text(
        markdown_drift(cycle_id, lotes_novos, tabela_drift, drift_resumo, referencia.y.mean(), novo.y.mean()), encoding="utf-8"
    )
    display(tabela_drift.head(12))
    tabela_drift.set_index("feature")["psi"].sort_values().plot.barh(figsize=(8, 5), title="PSI por atributo (lote novo x referência)")
    plt.axvline(cfg["drift"]["psi_alerta"], color="red", linestyle="--")
    plt.show()
salvar_json(pasta_ciclo / "drift.json", drift_resumo)
print("ALERTA DE DRIFT" if drift_resumo["alerta"] else "Sem alerta de drift", "-", drift_resumo.get("features_em_alerta", drift_resumo.get("motivo")))

Sem alerta de drift - ciclo inicial ou sem lote novo


## Persistência do ciclo e registro no MLflow

In [7]:
import mlflow
from src.rastreamento import iniciar_run_ciclo, run_etapa

processado = caminho(cfg, cfg["dados"]["diretorio_processado"])
processado.mkdir(parents=True, exist_ok=True)
treino.to_csv(processado / "treino.csv", index=False)
validacao.to_csv(processado / "validacao.csv", index=False)
teste.to_csv(processado / "teste.csv", index=False)

salvar_json(pasta_ciclo / "diagnostico.json", {"antes": diagnostico_antes, "depois": diagnostico_depois})
contexto = atualizar_contexto(
    cfg, cycle_id,
    cycle_id=cycle_id,
    data_version=data_version,
    lotes_novos=lotes_novos,
    lotes_anteriores=lotes_anteriores,
    lotes_pool=lotes_pool,
    n_treino=len(treino),
    n_validacao=len(validacao),
    n_teste=len(teste),
    qualidade_ok=True,
    drift_alerta=bool(drift_resumo["alerta"]),
)
contexto = atualizar_contexto(cfg, cycle_id, mlflow_parent_run_id=iniciar_run_ciclo(cfg, contexto))

with run_etapa(cfg, contexto, "etl"):
    mlflow.log_params({"lotes_novos": ",".join(lotes_novos) or "nenhum", "fracao_validacao": cfg["particionamento"]["fracao_validacao"]})
    mlflow.log_metrics({
        "linhas_pool": len(pool), "linhas_treino": len(treino), "linhas_validacao": len(validacao),
        "taxa_positiva_treino": treino.y.mean(), "taxa_positiva_validacao": validacao.y.mean(),
        "duplicatas_removidas": relatorio_limpeza["duplicatas_removidas"],
        "drift_alerta": int(drift_resumo["alerta"]), "drift_psi_maximo": drift_resumo.get("psi_maximo", 0.0),
    })
    for arquivo in ["qualidade_dados.json", "diagnostico.json", "drift.json", "drift.csv"]:
        if (pasta_ciclo / arquivo).exists():
            mlflow.log_artifact(str(pasta_ciclo / arquivo))

print(json.dumps(contexto, indent=2, ensure_ascii=False))

2026/11/23 12:13:08 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/23 12:13:08 INFO mlflow.store.db.utils: Updating database tables


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step


INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags


... (27 outputs left out)

INFO  [alembic.runtime.migration] Running upgrade 4465047574b1 -> f5a4f2784254, increase run tag value limit to 8000


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


{
  "cycle_id": "c20261123T121302Z-gh45172042466",
  "data_version": "v001-a59ff626",
  "lotes_novos": [
    "lote_001.csv"
  ],
  "lotes_anteriores": [],
  "lotes_pool": [
    "lote_001.csv"
  ],
  "n_treino": 20715,
  "n_validacao": 5227,
  "n_teste": 4119,
  "qualidade_ok": true,
  "drift_alerta": false,
  "mlflow_parent_run_id": "ed8c71a68d5c4a6abcf0c4dc260d6f54"
}
